# 🌍 Climatology Engine – Tutorial جامع

**یک راهنمای گام‌به‌گام برای کار با موتور اقلیم‌شناسی**

این نوت‌بوک تمام مراحل کار با **Climatology Engine** را از صفر تا صد نشان می‌دهد:
- بارگذاری داده
- برازش توزیع‌ها با معماری پلاگین
- انتخاب بهترین مدل با AICc
- کیفیت‌سنجی (Quality Flag)
- عدم‌قطعیت با Bootstrap
- ذخیره و تحلیل خروجی Zarr
- رسم نمودارها و نقشه‌ها

---

## 📦 ۱. نصب وابستگی‌ها و تنظیم مسیر

ابتدا مطمئن شوید که پروژه به‌درستی نصب شده است. اگر از محیط مجازی استفاده می‌کنید، آن را فعال کنید.

In [ ]:
# اگر در ریشه پروژه نیستید، مسیر را تنظیم کنید
import sys
import os

# مسیر پروژه را به sys.path اضافه کنید (در صورت نیاز)
project_root = os.path.abspath('..')  # اگر نوت‌بوک در پوشه notebooks/ است
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    print(f"✅ مسیر پروژه اضافه شد: {project_root}")

# بررسی اینکه آیا ماژول‌های پروژه قابل دسترس هستند
try:
    import core
    import plugins
    print("✅ ماژول‌های پروژه به‌درستی بارگذاری شدند.")
except ImportError as e:
    print(f"❌ خطا در بارگذاری ماژول‌ها: {e}")
    print("لطفاً مطمئن شوید که در مسیر درست قرار دارید.")

In [ ]:
# وارد کردن کتابخانه‌های اصلی
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# تنظیمات نمایش
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.2)
%matplotlib inline

print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")
print(f"Xarray version: {xr.__version__}")

## 📂 ۲. بارگذاری داده نمونه

پروژه شامل ۱۰ ایستگاه نمونه در پوشه `sample_data/` است. هر فایل شامل داده‌های روزانه `tmin`, `tmean`, `tmax` برای ۳۰ سال (۱۰۹۵۰ روز) است.

In [ ]:
# مسیر داده نمونه
sample_dir = os.path.join(project_root, 'sample_data')

# لیست همه فایل‌های ایستگاه
station_files = sorted([f for f in os.listdir(sample_dir) if f.endswith('.csv')])
print(f"✅ تعداد ایستگاه‌های نمونه: {len(station_files)}")
print(f"   فایل‌ها: {station_files[:5]}...")

# بارگذاری اولین ایستگاه
station_data = pd.read_csv(os.path.join(sample_dir, station_files[0]))
print(f"\n📊 شکل داده: {station_data.shape}")
print(f"   ستون‌ها: {station_data.columns.tolist()}")
print(f"\nنمونه داده:")
station_data.head()

In [ ]:
# نمایش داده‌های یک سال
data = station_data.values
n_days = 365

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(data[:n_days, 0], color='blue', alpha=0.7, linewidth=1.5)
axes[0].set_ylabel('tmin (°C)')
axes[0].set_title('دمای کمینه - ایستگاه ۱ (سال اول)')
axes[0].grid(True, alpha=0.3)

axes[1].plot(data[:n_days, 1], color='green', alpha=0.7, linewidth=1.5)
axes[1].set_ylabel('tmean (°C)')
axes[1].set_title('دمای میانگین')
axes[1].grid(True, alpha=0.3)

axes[2].plot(data[:n_days, 2], color='red', alpha=0.7, linewidth=1.5)
axes[2].set_xlabel('Day of Year')
axes[2].set_ylabel('tmax (°C)')
axes[2].set_title('دمای بیشینه')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 🔌 ۳. بارگذاری پلاگین‌های توزیع

موتور از معماری پلاگین استفاده می‌کند. توزیع‌ها در پوشه `plugins/distributions/` قرار دارند و به‌صورت خودکار بارگذاری می‌شوند.

In [ ]:
from core.engine.plugin_loader import load_plugins

plugins = load_plugins()
print(f"✅ تعداد توزیع‌های بارگذاری شده: {len(plugins)}")
for code, dist in plugins.items():
    print(f"   [{code}] {dist.name} (params: {dist.params})")

# ذخیره توزیع‌ها در دیکشنری جداگانه
distributions = {dist.name: dist for dist in plugins.values()}

## 📊 ۴. برازش یک توزیع

ابتدا یک توزیع نرمال را روی داده‌های یک سال برازش می‌دهیم.

In [ ]:
# انتخاب داده tmean برای یک سال
data_year = data[:365, 1]  # tmean
print(f"تعداد داده‌ها: {len(data_year)}")
print(f"میانگین: {np.mean(data_year):.2f}°C")
print(f"انحراف معیار: {np.std(data_year):.2f}°C")

# برازش توزیع نرمال
normal_dist = distributions['Normal']
result = normal_dist.fit(data_year)

print("\n📈 نتایج برازش توزیع نرمال:")
for key, value in result.items():
    print(f"   {key}: {value:.4f}" if isinstance(value, float) else f"   {key}: {value}")

In [ ]:
# رسم هیستوگرام و منحنی توزیع نرمال
from scipy.stats import norm

mu = result['p1']
sigma = result['p2']

fig, ax = plt.subplots(figsize=(10, 6))

# هیستوگرام داده
ax.hist(data_year, bins=30, density=True, alpha=0.6, color='blue', edgecolor='black', label='داده')

# منحنی نرمال
x = np.linspace(min(data_year), max(data_year), 200)
pdf = norm.pdf(x, mu, sigma)
ax.plot(x, pdf, 'r-', linewidth=2.5, label=f'Normal(μ={mu:.2f}, σ={sigma:.2f})')

ax.set_xlabel('دما (°C)')
ax.set_ylabel('چگالی احتمال')
ax.set_title('برازش توزیع نرمال روی داده‌های دما')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 🔬 ۵. برازش همه توزیع‌ها و انتخاب بهترین مدل

حالا همه توزیع‌های موجود را روی داده برازش می‌دهیم و بهترین مدل را بر اساس **AICc** انتخاب می‌کنیم.

In [ ]:
# برازش همه توزیع‌ها
results_all = {}
for name, dist in distributions.items():
    try:
        res = dist.fit(data_year)
        results_all[name] = res
        print(f"✅ {name}: AICc = {res.get('aicc', np.nan):.2f}")
    except Exception as e:
        print(f"❌ {name}: خطا - {str(e)}")
        results_all[name] = None

In [ ]:
# انتخاب بهترین مدل (کمترین AICc)
valid_results = {k: v for k, v in results_all.items() if v is not None and 'aicc' in v}
best_name = min(valid_results, key=lambda x: valid_results[x]['aicc'])
best_result = valid_results[best_name]

print("=" * 60)
print(f"🏆 بهترین توزیع: {best_name}")
print(f"   AICc: {best_result['aicc']:.4f}")
print(f"   BIC: {best_result.get('bic', np.nan):.4f}")
print(f"   Log-likelihood: {best_result.get('loglik', np.nan):.4f}")
print("=" * 60)

# نمایش جدول مقایسه
comparison_df = pd.DataFrame([{
    'Distribution': name,
    'AICc': res['aicc'],
    'BIC': res.get('bic', np.nan),
    'ΔAICc': res['aicc'] - best_result['aicc']
} for name, res in valid_results.items()])

comparison_df = comparison_df.sort_values('AICc')
comparison_df

In [ ]:
# رسم مقایسه AICc توزیع‌ها
fig, ax = plt.subplots(figsize=(10, 6))

colors = ['#2ecc71' if name == best_name else '#e74c3c' for name in comparison_df['Distribution']]
bars = ax.barh(comparison_df['Distribution'], comparison_df['AICc'], color=colors, alpha=0.7)

ax.axvline(best_result['aicc'], color='black', linestyle='--', linewidth=1.5, alpha=0.5, label=f'Best: {best_name}')
ax.set_xlabel('AICc')
ax.set_title('مقایسه AICc توزیع‌های مختلف')
ax.legend()
ax.grid(True, alpha=0.3, axis='x')

for i, (name, aicc) in enumerate(zip(comparison_df['Distribution'], comparison_df['AICc'])):
    ax.text(aicc + 1, i, f'{aicc:.1f}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

## 🛡️ ۶. کیفیت‌سنجی (Quality Flag)

سیستم Quality Flag به‌صورت خودکار کیفیت برازش را ارزیابی می‌کند.

In [ ]:
from core.quality.quality_flag import QualityFlag

# ارزیابی کیفیت برای بهترین توزیع
flags = QualityFlag.evaluate(best_result, data_year)

flag_names = {
    QualityFlag.PASS: '✅ PASS',
    QualityFlag.LOW_SAMPLE: '⚠️ LOW_SAMPLE',
    QualityFlag.NO_CONVERGENCE: '❌ NO_CONVERGENCE',
    QualityFlag.OUTLIER: '⚠️ OUTLIER',
    QualityFlag.HIGH_AICC: '⚠️ HIGH_AICC',
    QualityFlag.BAD_SKEW: '⚠️ BAD_SKEW',
    QualityFlag.NAN_INPUT: '❌ NAN_INPUT',
    QualityFlag.INF_INPUT: '❌ INF_INPUT',
}

print(f"📋 کیفیت برازش برای {best_name}:")
for flag in flags:
    print(f"   {flag_names.get(flag, f'UNKNOWN ({flag})')}")

## 📈 ۷. عدم‌قطعیت با Bootstrap

برای تخمین عدم‌قطعیت پارامترها از روش Bootstrap استفاده می‌کنیم.

In [ ]:
from core.uncertainty.bootstrap import bootstrap_fit

# تعریف تابع برازش برای bootstrap
def fit_func(data):
    dist = distributions['Normal']
    return dist.fit(data)

# اجرای bootstrap
n_bootstrap = 50  # برای سرعت، تعداد کمتر (در عمل ۱۰۰-۱۰۰۰)
cis = bootstrap_fit(data_year, fit_func, n_bootstrap=n_bootstrap, confidence=0.95)

print("📊 عدم‌قطعیت پارامترها (95% CI):")
print("=" * 50)
for param, ci in cis.items():
    print(f"{param}:")
    print(f"   mean: {ci['mean']:.4f}")
    print(f"   lower: {ci['lower']:.4f}")
    print(f"   upper: {ci['upper']:.4f}")
    print("   ")

In [ ]:
# رسم فاصله اطمینان Bootstrap
fig, ax = plt.subplots(figsize=(8, 6))

params = list(cis.keys())
means = [cis[p]['mean'] for p in params]
lowers = [cis[p]['lower'] for p in params]
uppers = [cis[p]['upper'] for p in params]

y_pos = np.arange(len(params))
ax.errorbar(means, y_pos, xerr=[means - np.array(lowers), np.array(uppers) - means],
            fmt='o', color='blue', capsize=5, capthick=2, elinewidth=2, markersize=10)

ax.set_yticks(y_pos)
ax.set_yticklabels(params)
ax.set_xlabel('مقدار پارامتر')
ax.set_title('فاصله اطمینان ۹۵% پارامترها (Bootstrap)')
ax.axvline(0, color='black', linestyle='-', alpha=0.2)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 💾 ۸. ذخیره نتایج در Zarr

نتایج برازش را در یک فایل Zarr ذخیره می‌کنیم.

In [ ]:
# ایجاد یک دیتاست ساده برای ذخیره نتایج
n_stations = len(station_files)
n_days = 365

# ایجاد آرایه‌های خالی
best_dist = np.full((n_stations, n_days), -1, dtype=np.int32)
mean_vals = np.full((n_stations, n_days), np.nan, dtype=np.float32)
std_vals = np.full((n_stations, n_days), np.nan, dtype=np.float32)

print(f"📊 ایجاد آرایه‌ها با شکل: ({n_stations}, {n_days})")
print("   (اینجا فقط یک نمونه ساده است – در عمل، پردازش کامل انجام می‌شود)")

In [ ]:
# ذخیره در Zarr
zarr_path = os.path.join(project_root, 'nature_output', 'test_output.zarr')
os.makedirs(os.path.dirname(zarr_path), exist_ok=True)

# ایجاد دیتاست xarray
ds = xr.Dataset(
    data_vars={
        'best_dist': (('station', 'day'), best_dist),
        'mean': (('station', 'day'), mean_vals),
        'std': (('station', 'day'), std_vals),
    },
    coords={
        'station': np.arange(n_stations),
        'day': np.arange(n_days),
    }
)

# ذخیره
ds.to_zarr(zarr_path, mode='w', consolidated=False)
print(f"✅ فایل Zarr در {zarr_path} ذخیره شد.")
print(f"   ابعاد: {ds.dims}")
print(f"   متغیرها: {list(ds.data_vars)}")

In [ ]:
# بازخوانی Zarr
ds_loaded = xr.open_zarr(zarr_path, consolidated=False)
print("✅ فایل Zarr بازخوانی شد.")
print(f"   ابعاد: {ds_loaded.dims}")
print(f"   متغیرها: {list(ds_loaded.data_vars)}")
print(f"\nنمونه داده best_dist:")
print(ds_loaded['best_dist'].values[:5, :5])

## 📊 ۹. تحلیل و رسم نتایج

اکنون می‌توانیم نتایج را تحلیل و رسم کنیم.

In [ ]:
# تابع برای محاسبه توزیع غالب در هر روز
def get_dominant_distribution(best_dist_array):
    n_days = best_dist_array.shape[1]
    dominant = np.zeros(n_days, dtype=int)
    for day in range(n_days):
        counts = np.bincount(best_dist_array[:, day].astype(int))
        if len(counts) > 0:
            dominant[day] = np.argmax(counts)
        else:
            dominant[day] = -1
    return dominant

# برای داده‌های نمونه، یک best_dist تصادفی می‌سازیم
np.random.seed(42)
mock_best = np.random.choice([0, 1, 2, 3], size=(n_stations, n_days))
dominant = get_dominant_distribution(mock_best)

# توزیع‌های غالب
dist_names = {0: 'Normal', 1: 'Skew', 2: 'Bimodal', 3: 'Pearson'}
dominant_names = [dist_names.get(d, 'Unknown') for d in dominant]

fig, ax = plt.subplots(figsize=(14, 5))
unique, counts = np.unique(dominant_names, return_counts=True)
bars = ax.bar(unique, counts, color=['#3498db', '#2ecc71', '#e74c3c', '#f39c12'])

ax.set_xlabel('Distribution')
ax.set_ylabel('Number of Days')
ax.set_title('توزیع غالب در طول سال')
ax.grid(True, alpha=0.3, axis='y')

for bar, count in zip(bars, counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, str(count),
            ha='center', va='bottom', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# رسم نقشه توزیع‌ها در طول سال
fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(mock_best, aspect='auto', cmap='viridis', interpolation='none')

ax.set_xlabel('Day of Year')
ax.set_ylabel('Station')
ax.set_title('نقشه توزیع‌ها برای ایستگاه‌ها و روزهای مختلف')

cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(['Normal', 'Skew', 'Bimodal', 'Pearson'])

plt.tight_layout()
plt.show()

## 🧪 ۱۰. اجرای تست‌ها

برای اطمینان از صحت عملکرد، تست‌های واحد را اجرا می‌کنیم.

In [ ]:
!pytest tests/test_distributions.py -v --tb=short 2>&1 | head -30

## 🚀 ۱۱. اجرای کامل موتور

برای اجرای کامل موتور روی داده‌های واقعی، از دستور زیر استفاده کنید (در صورت وجود فایل `main.py`):

In [ ]:
# اجرای موتور اصلی
# !python main.py

print("""
برای اجرای کامل موتور:
  1. فایل config.yaml را با مسیرهای خود تنظیم کنید.
  2. در ترمینال اجرا کنید: python main.py

پارامترهای مهم:
  - block_size: تعداد ایستگاه‌ها در هر بلوک (پیش‌فرض ۱۰۰۰)
  - use_extreme_values: true/false (حالت حدی)
  - n_points_max: حداکثر تعداد نقاط (پیش‌فرض ۴۰۰۰۰)
""")

## 📋 ۱۲. جمع‌بندی

در این نوت‌بوک با موارد زیر آشنا شدید:

✅ بارگذاری داده‌های نمونه
✅ برازش توزیع‌ها با معماری پلاگین
✅ انتخاب بهترین مدل با AICc
✅ کیفیت‌سنجی با Quality Flag
✅ عدم‌قطعیت با Bootstrap
✅ ذخیره و بازخوانی Zarr
✅ تحلیل و رسم نتایج

---

**مراحل بعدی:**
1. تنظیم `config.yaml` برای داده‌های خود
2. اجرای `python main.py` برای پردازش کامل
3. تحلیل خروجی با `analyze_distributions.py`
4. اضافه کردن توزیع‌های جدید در `plugins/distributions/`

---

**منابع مفید:**
- 📖 [مستندات کامل](https://climatology-engine.readthedocs.io/)
- 🐛 [گزارش مشکل](https://github.com/AminFazlKazemi/ClimateProcessingEngine/issues)
- 💬 [گفتگو و بحث](https://github.com/AminFazlKazemi/ClimateProcessingEngine/discussions)

In [ ]:
print("=" * 60)
print("🎉 نوت‌بوک با موفقیت کامل شد!")
print("=" * 60)
print(f"📁 پوشه کاری: {os.getcwd()}")
print(f"📂 داده نمونه: {sample_dir}")
print(f"💾 خروجی Zarr: {zarr_path}")
print("=" * 60)